In [ ]:
# Import des bibliothèques nécessaires au code
# Import of necessary libraries 
import geopandas as gpd  #Analyse vecteur
from shapely.geometry import Point, Polygon #Analyse vecteur
from sklearn.preprocessing import MinMaxScaler #Analyse statistiques
import numpy as np  #Manipulation de données 
import pandas as pd  #Manipulation de données via des "tableaux"
import rasterio  #Analyse raster
from rasterio.warp import calculate_default_transform, reproject, Resampling  #Extensions de rasterio
import glob  #Traitement de fichier
import os  #Interaction avec le système d'exploitation
from rasterio.merge import merge  #Ajoute la fonction "merge" de rasterio afin de fusionner les tuiles rasters

In [ ]:
# Transformation des couches shapefiles en GeoDataFrames, semblables à des tableaux avec géographie, utilisable par GéoPandas
# Transform shapefiles layers into GeoDataFrames, seems like table with a geographical component, use with GeoPandas
Commune = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/Avignon.shp")
Cadastre = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/cadastre.shp")
Batiments = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/batiments.shp")
Surface_hydro = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/Surface_hydro.shp")
Troncon_route = gpd.read_file("C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/trancon_route.shp")

In [ ]:
# Début de la fusion des tuiles RGE Alti 5m, chemin vers les fichiers d'origine et de destination
# Starting the merging of the 'RGE Alti 5m' files. File path for the RGE files and the writing path (sortie)
dossier_tuiles = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE\RGEALTI_2-0_5M_ASC_LAMB93-IGN69_D084_2022-12-16\RGEALTI\1_DONNEES_LIVRAISON_2023-01-00223\RGEALTI_MNT_5M_ASC_LAMB93_IGN69_D084/*.*"
sortie = r"C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/RGE/DME.tif"

In [ ]:
# Fonction merge pour fusionner les tuiles RGE Alti 5m
# Merge function to merge the RGE files (altitude)
mosaic, out_trans = merge(glob.glob(dossier_tuiles))

In [ ]:
# Copie ("out_meta") les paramètres du premier fichier de dossier ('dossier_tuiles')
# ("out_meta") copy : parameters of the first file in the folder ('dossier_tuiles')
with rasterio.open(glob.glob(dossier_tuiles)[0]) as dem:
    out_meta = dem.meta.copy()

In [ ]:
# Modifie les paramètres de "out_meta" : Format du fichier, hauteur, épaisseur et la géographie
# Change the parameters of "out_meta" : File format, height, width and geography
out_meta.update({
    'driver': 'GTiff',
    'height': mosaic.shape[1],
    'width': mosaic.shape[2],
    'transform': out_trans,
})

In [ ]:
# Ecriture du fichier, format écriture
# File writing, writing format 'w'
with rasterio.open(sortie, 'w', **out_meta) as dem:
    dem.write(mosaic)

In [ ]:
# Début du calcul de la pente par GDAL, variable du fichier d'origine et de destination
# Starting of the slope calculation by GDAL. File paths
dem_chemin = r"C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/RGE/DME.tif"
slope_chemin = r"C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/RGE/slope.tif"

In [ ]:
# Méthode numéro 1 : Calcul de la pente par l'API GDAL.  !! Ne marche pas sur mon code, sûrement
# par l'écriture du fichier qui malfonctionne. Nous allons donc calculer le code par une commande bash

# First method : Calculation with GDAL API, but doesn't work here, we'll use a bash command with subprocess

#from osgeo import gdal  (Import de GDAL)

#dem = gdal.Open(dem_chemin) Fariable GDAL de "dem_chemin"

# options = gdal.DEMProcessingOptions(   #Création des options pour le calcul de la pente
    #format='GTiff',
    #slopeFormat='degree',
    #computeEdges=True,
    #zeroForFlat=True,
    #alg='Horn',
#)

#gdal.DEMProcessing(   #Calcul de la pentre avec "slope"
    #destName=slope_chemin,
    #srcDS=dem,
    #processing='slope',
    #options=options
#)


In [ ]:
# Utilise subprocess pour exécuter une commande bash
# Subprocess for run the bash command
import subprocess

In [ ]:



# Ecriture du chemin des fichiers
# File path writing
dme_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE\DME.tif"
slope_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE\slope.tif"

# Commande GDAL avec l'algorithme "slope"
# GDAL command with the algorithm "slope"
cmd = [
    "gdaldem",
    "slope",
    dme_path,
    slope_path,
    "-of", "GTiff",
    "-s", "1",
    "-alg", "Horn"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['gdaldem', 'slope', 'C:\\Users\\edwin\\Desktop\\QGIS TUTORIAL\\Schema_de_traitement_Python\\RGE\\DME.tif', 'C:\\Users\\edwin\\Desktop\\QGIS TUTORIAL\\Schema_de_traitement_Python\\RGE\\slope.tif', '-of', 'GTiff', '-s', '1', '-alg', 'Horn'], returncode=0)

In [ ]:
# Création du fichier "calculated", un raster binaire avec nodata = Pente inférieure à 15 degrès, et 1= Pente supérieure à 15 degrès
# Creation of the file "calculated", a binary with nodata = Slope < 15 degree ; 1= Slope > 15 degree


slope_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/slope.tif"
calculated_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated.tif"


cmd = [
    "python",
    "-m", "osgeo_utils.gdal_calc",
    "-A", slope_path,
    "--outfile=" + calculated_path,
    "--calc=A>15",
    "--type=Byte",
    "--NoDataValue=0",
    "--overwrite"
]


subprocess.run(cmd, check=True)


CompletedProcess(args=['python', '-m', 'osgeo_utils.gdal_calc', '-A', 'C:\\Users\\edwin\\Desktop\\QGIS TUTORIAL\\Schema_de_traitement_Python\\RGE/slope.tif', '--outfile=C:\\Users\\edwin\\Desktop\\QGIS TUTORIAL\\Schema_de_traitement_Python\\RGE/calculated.tif', '--calc=A>15', '--type=Byte', '--NoDataValue=0', '--overwrite'], returncode=0)

In [ ]:
# La méthode de vectorisaion par GDAL. Fonctionne mais est très lente
# Vectorization with GDAL, works but really slowly

#calculated_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated.tif"
##vectorized_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/vectorized.gpkg"

#cmd = [
    #"python",
    #r"C:\Users\edwin\miniconda3\envs\geo\Scripts\gdal_polygonize.py",
    #calculated_path,
    #"-f", "GPKG",
    #"-b", "1",
    #vectorized_path
#]

#subprocess.run(cmd, check=True)

In [ ]:
# Avant de vectoriser notre raster "calculated", il faut le reprojeter en 2154, avec rasterio
# Before starting to vectorize, we have to reproject in 2154 "calculated" with rasterio

from rasterio.warp import calculate_default_transform, reproject, Resampling

# Chemins des fichiers
# File paths
calculated_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated.tif"
calculated_2154_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated_2154.tif"


with rasterio.open(calculated_path) as src:
    raster = src.read(1)
    profile = src.profile

    # Définition du EPSG : 2154 
    # EPSG : 2154
    profile.update(crs='EPSG:2154')

    # Calculer la transformation pour la reprojection
    # Transformation calculation to reproject
    transform, width, height = calculate_default_transform(
        src.crs, profile['crs'], src.width, src.height, *src.bounds
    )
    profile.update(transform=transform, width=width, height=height)

    # Créer et sauvegarder le raster reprojeté
    # Create and save the reproject raster
    with rasterio.open(calculated_2154_path, 'w', **profile) as dst:
        reproject(
            source=raster,
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=2154,
            dst_transform=transform,
            dst_crs='EPSG:2154',
            resampling=Resampling.nearest
        )


In [ ]:
# Nous allons désormais effectuer la vectorisation avec rasterio. Puis le sauvegarder en gpkg
# Vectorization with rasterio, followed by its transformation in gpkg

from rasterio.features import shapes


raster_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/calculated_2154.tif"
output_path = r"C:\Users\edwin\Desktop\QGIS TUTORIAL\Schema_de_traitement_Python\RGE/vectorized.gpkg"

# Charger le raster
# Raster loading
with rasterio.open(raster_path) as src:
    raster = src.read(1)
    transform = src.transform
    crs = src.crs

# Polygoniser  avec la valeur 1
# Polygonize with value 1
polygons = shapes(raster, mask=(raster == 1), transform=transform)

# Créer le GeoDataFrame
# Creation of the GDF
pente_gdf = gpd.GeoDataFrame.from_features(
    [{"properties": {"valeur": 1}, "geometry": geom} for geom, val in polygons],
    crs=src.crs
)


# Sauvegarder en GeoPackage
# Save in GeoPackage
pente_gdf.to_file(output_path, driver="GPKG")

In [ ]:
# Reprojeter les GeoDataFrames en Lambert93
# Reproject the GeoDataFrames in Lambert93
Commune = Commune.to_crs("EPSG:2154")
Cadastre = Cadastre.to_crs("EPSG:2154")
Batiments = Batiments.to_crs("EPSG:2154")
Surface_hydro = Surface_hydro.to_crs("EPSG:2154")
Troncon_route = Troncon_route.to_crs("EPSG:2154")
pente_gdf = pente_gdf.to_crs("EPSG:2154")

In [ ]:
# Extraire les entités des GeoDataFrames qui se superposent avec la commune
# Extraction of the GeoDataFrames who clip the city geometry (Commune)
commune_cadastre = Cadastre.clip(Commune)
commune_batiments = Batiments.clip(Commune)
commune_hydro = Surface_hydro.clip(Commune)
commune_route = Troncon_route.clip(Commune)
commune_pente = pente_gdf.clip(Commune)

In [ ]:
# Regrouper les entités du GDF "commune_pente_dissolve"
# Dissolve of the slope GDF 
commune_pente_dissolve = commune_pente.dissolve()


In [ ]:
# Créer le buffer de 20 mètres autour des routes, puis transformer le GeoSeries créer en GeoDataFrame
# 20 meters buffer around roads
route_buffer = commune_route.geometry.buffer(20)
route_buffer_gdf = gpd.GeoDataFrame(geometry=route_buffer, crs=route_buffer.crs)

In [ ]:
# Créer notre GeoDataFrame "contraintes" qui regroupe toutes les contraintes, puis fusionner les entités 
# Creation of our constraints file "contraintes", who regroup all the constraints (roads, water...)
contraintes = gpd.GeoDataFrame(pd.concat([commune_batiments, commune_hydro, route_buffer_gdf, commune_pente_dissolve], ignore_index=True))
contraintes_dissolve = contraintes.dissolve()

In [ ]:
# Fusionner les entités du GDF "commune_cadastre" pour les prochains calculs
# Cadastre GDF dissolve for our next calculations
cadastre_dissolve = commune_cadastre.dissolve()

In [ ]:
# Création du GDF "Domaine_Public" = Les terrains de la commune qui ne se superposent pas avec le cadastre
# Public Domain GDF creation = territories who are not over a cadastre geometry
Domaine_public = Commune.geometry.difference(cadastre_dissolve)
Domaine_public_gdf = gpd.GeoDataFrame(geometry=Domaine_public, crs=Domaine_public.crs)

In [ ]:
# Pour finir, créer le GDF "dp_Disponible" = Les terrains de Domaine Public qui ne se surperposent pas avec les contraintes
# Finally, create the available public domain = public terrains not subject to constraints
dp_Disponible = Domaine_public_gdf.geometry.difference(contraintes_dissolve)
dp_Disponible_gdf = gpd.GeoDataFrame(geometry=dp_Disponible, crs=dp_Disponible.crs)

In [ ]:
# Création du GPKG
# GPKG creation
dp_Disponible_gdf.to_file('C:/Users/edwin/Desktop/QGIS TUTORIAL/Schema_de_traitement_Python/Final/Domaine_Public_Disponible.gpkg', driver='GPKG', layer='name')